# 00b City data: central Khartoum from Overture Maps

Roads and buildings for the city-scale maps in `05b_city_and_relief.ipynb`. Reads Overture's open GeoParquet files straight from cloud storage, keeping only the box below (needs internet, a minute or two), then saves to `data/processed/`. The files are gitignored.

Install once: `pip install -e .[demo]` (adds the overturemaps client).

Overture combines OpenStreetMap with other open sources. Credit on any map made from it: Overture Maps Foundation, OpenStreetMap contributors (ODbL).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from overturemaps import core

OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)
# Central Khartoum and the Nile confluence: west, south, east, north
BBOX = (32.49, 15.57, 32.57, 15.63)
# UTM zone 36N, for lengths and areas in metres
UTM = 32636


def download(kind):
    # One Overture type inside the box, in EPSG:4326
    gdf = core.geodataframe(kind, bbox=BBOX)
    return gdf if gdf.crs is not None else gdf.set_crs(4326)


def primary_name(names):
    # Overture keeps names as a struct; the main name is under 'primary'
    return names.map(lambda d: d.get("primary") if isinstance(d, dict) else None)

In [2]:
# Roads: every transport segment in the box
raw_roads = download("segment")
raw_roads.shape

(10743, 21)

In [3]:
# Keep roads only, group Overture classes into seven classes plus other
CLASSES = {
    "motorway": "highway", "trunk": "highway",
    "primary": "primary", "secondary": "secondary", "tertiary": "tertiary",
    "residential": "residential", "living_street": "residential", "unclassified": "residential",
    "service": "service",
    "pedestrian": "path", "footway": "path", "steps": "path", "path": "path", "track": "path",
    "cycleway": "path", "bridleway": "path",
}
RANK = {"highway": 7, "primary": 6, "secondary": 5, "tertiary": 4, "residential": 3, "service": 2, "path": 1, "other": 0}
roads = raw_roads[(raw_roads["subtype"] == "road") & raw_roads.geom_type.isin(["LineString", "MultiLineString"])].copy()
roads["highway"] = roads["class"].astype("string")
roads["road_class"] = roads["highway"].map(CLASSES).fillna("other")
roads["rank"] = roads["road_class"].map(RANK)
roads["name"] = primary_name(roads["names"]) if "names" in roads else None
roads["length_m"] = roads.to_crs(UTM).length.round(1)
roads = roads[["name", "highway", "road_class", "rank", "length_m", "geometry"]].reset_index(drop=True)
roads.to_file(OUT / "khartoum_roads.gpkg")
roads["road_class"].value_counts()

road_class
residential    6787
tertiary       1429
service         986
secondary       768
primary         357
highway         221
path            154
other             4
Name: count, dtype: int64

In [4]:
# Buildings: every footprint in the box
raw_buildings = download("building")
raw_buildings.shape

(71223, 24)

In [5]:
# Heights: mapped height, else 3 m per mapped floor, else 6 m (two storeys), with the source kept
b = raw_buildings[raw_buildings.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
height = pd.to_numeric(b["height"], errors="coerce") if "height" in b else pd.Series(np.nan, index=b.index)
floors = pd.to_numeric(b["num_floors"], errors="coerce") if "num_floors" in b else pd.Series(np.nan, index=b.index)
b["height_m"] = height.fillna(floors * 3).fillna(6.0).round(1)
b["height_source"] = np.select([height.notna(), floors.notna()], ["mapped", "floors"], "assumed")
b["levels"] = floors
b["kind"] = b["class"].astype("string").fillna("unspecified") if "class" in b else "unspecified"
b["name"] = primary_name(b["names"]) if "names" in b else None
b["area_m2"] = b.to_crs(UTM).area.round(1)
buildings = b[["name", "kind", "levels", "height_m", "height_source", "area_m2", "geometry"]].reset_index(drop=True)
buildings.to_file(OUT / "khartoum_buildings.gpkg")
buildings["height_source"].value_counts()

height_source
assumed    70987
floors       126
mapped       110
Name: count, dtype: int64

In [6]:
# Quick look: buildings under roads coloured by class
import kharita as kh

m = kh.Map()
m.add(buildings, "fill-slate-300 stroke-0.5 stroke-slate-400 legend-none")
m.add(roads, "stroke-[road_class] ramp-dark2 stroke-1.5", legend={"title": "Road class"})
m